In [1]:
import csv, json
import pandas as pd
from pathlib import Path

In [4]:
from collections import defaultdict

base_ds = Path(r"C:\Users\sakoehn\Desktop\JGI\DataStructures")

with open(base_ds / r"Enzyme\enz_db.json") as f:
    cog_db = json.load(f)

by_tp = defaultdict(dict)

for gene_id, info in cog_db.items():
    tps = info.get("timepoint", [])
    for tp in tps:
        by_tp[tp][gene_id] = info  # same structure, just scoped to this tp

out_dir = base_ds / r"Enz_byTP"
out_dir.mkdir(parents=True, exist_ok=True)

for tp, subdict in by_tp.items():
    out_path = out_dir / f"enz_db_{tp}.json"
    with open(out_path, "w") as f:
        json.dump(subdict, f)
    print("wrote", out_path, "entries:", len(subdict))

wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Enz_byTP\enz_db_3300060358.json entries: 143949
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Enz_byTP\enz_db_3300060359.json entries: 133320
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Enz_byTP\enz_db_3300060360.json entries: 130322
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Enz_byTP\enz_db_3300060361.json entries: 154005
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Enz_byTP\enz_db_3300060362.json entries: 190957
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Enz_byTP\enz_db_3300060363.json entries: 173702
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Enz_byTP\enz_db_3300060364.json entries: 135133
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Enz_byTP\enz_db_3300060365.json entries: 135831
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Enz_byTP\enz_db_3300060366.json entries: 137552
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Enz_byTP\enz_db_3300060367.json entries: 122756
wrote C:\Users\sakoe

In [5]:
import json
import pandas as pd
from pathlib import Path

# Base data-structures directory
base_ds = Path(r"C:\Users\sakoehn\Desktop\JGI\DataStructures")

# Sample names
with open(r"C:\Users\sakoehn\Desktop\JGI\sample_names.txt") as f:
    samples = [s.strip() for s in f]


In [7]:

# Output dir for final per-sample mapping tables
out_dir = base_ds / "Mappings_by_sample"
out_dir.mkdir(parents=True, exist_ok=True)

def _first_or_none(x):
    if not x:
        return None
    return x[0]

# Helpers: load per-sample dbs
def load_cog_for_sample(sample):
    path = base_ds / "COG_byTP" / f"cog_db_{sample}.json"
    with open(path) as f:
        return json.load(f)

def load_kegg_for_sample(sample):
    path = base_ds / "Kegg_byTP" / f"kegg_db_{sample}.json"
    with open(path) as f:
        return json.load(f)

def load_enz_for_sample(sample):
    path = base_ds / "Enz_byTP" / f"enz_db_{sample}.json"
    with open(path) as f:
        return json.load(f)

def load_tax_for_sample(sample):
    path = base_ds / "tax_db_byTP" / f"tax_db_{sample}.json"
    with open(path) as f:
        return json.load(f)

# Build simple lookups for per-sample dicts
def db_to_lookup_sample(db, sample, value_key):
    rows = []
    for gene_id, info in db.items():
        vals = info.get("lookupId", [])
        val = _first_or_none(vals)
        rows.append({"gene": gene_id, "sample": sample, value_key: val})
    return pd.DataFrame(rows)

def tax_lookup_for_sample(sample):
    tax_db = load_tax_for_sample(sample)
    rows = []
    for gene_id, info in tax_db.items():
        scaffs = info.get("scaffolds", [])
        tax_ids = info.get("lookupId", [])
        scaffold = _first_or_none(scaffs)
        taxonomic = _first_or_none(tax_ids)
        rows.append({
            "gene": gene_id,
            "sample": sample,
            "scaffold": scaffold,
            "taxonomic": taxonomic,
        })
    return pd.DataFrame(rows)

# Main per-sample loop
for sample in samples:
    print("processing sample", sample)

    # Load small per-sample dbs (files opened/closed inside loop)
    cog_db_s  = load_cog_for_sample(sample)
    kegg_db_s = load_kegg_for_sample(sample)
    enz_db_s  = load_enz_for_sample(sample)

    # base: genes present in this sample (use COG as primary list)
    genes = list(cog_db_s.keys())
    if not genes:
        print("no genes for", sample, "- skipping")
        continue

    base = pd.DataFrame({"gene": genes, "sample": sample})

    # Build lookups for this sample
    cog_lookup  = db_to_lookup_sample(cog_db_s,  sample, "cog")
    kegg_lookup = db_to_lookup_sample(kegg_db_s, sample, "kegg")
    enz_lookup  = db_to_lookup_sample(enz_db_s,  sample, "enzyme")
    tax_lookup  = tax_lookup_for_sample(sample)

    # Join everything on gene + sample
    table = base.merge(cog_lookup, on=["gene", "sample"], how="left")
    table = table.merge(kegg_lookup, on=["gene", "sample"], how="left")
    table = table.merge(enz_lookup, on=["gene", "sample"], how="left")
    table = table.merge(tax_lookup, on=["gene", "sample"], how="left")

    # Rename columns to final schema
    table = table.rename(columns={
        "gene": "gene_id",
        "scaffold": "scaffold_id",
        "taxonomic": "taxonomy",
    })

    # Write final per-sample mapping table
    out_path = out_dir / f"{sample}_mapping.tsv"
    table[["gene_id", "scaffold_id", "taxonomy", "kegg", "enzyme", "cog"]].to_csv(
        out_path, sep="\t", index=False
    )
    print("wrote", out_path, "rows:", len(table))

processing sample 3300060358
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Mappings_by_sample\3300060358_mapping.tsv rows: 342827
processing sample 3300060359
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Mappings_by_sample\3300060359_mapping.tsv rows: 322004
processing sample 3300060360
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Mappings_by_sample\3300060360_mapping.tsv rows: 322169
processing sample 3300060361
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Mappings_by_sample\3300060361_mapping.tsv rows: 382800
processing sample 3300060362
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Mappings_by_sample\3300060362_mapping.tsv rows: 404568
processing sample 3300060363
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Mappings_by_sample\3300060363_mapping.tsv rows: 380146
processing sample 3300060364
wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Mappings_by_sample\3300060364_mapping.tsv rows: 307277
processing sample 3300060365
wrote C:\Users\sakoehn\Desktop\JG